# initialiastion du projet de detection de la pneumonie 

In [ ]:
import tensorflow as tf # Bibliothèque principale pour la création et l'entraînement de modèles de Deep Learning
from tensorflow.keras import layers, models # Importation des outils pour construire les couches (Conv2D, Dense) et la structure (Sequential)
from tensorflow.keras.utils import plot_model # Outil pour générer un schéma visuel de l'architecture de votre réseau de neurones
import matplotlib.pyplot as plt # Bibliothèque pour tracer les graphiques (Accuracy, Loss) et afficher les images médicales
import numpy as np # Outil de calcul numérique pour manipuler les données (tableaux d'images et de labels)
from sklearn.metrics import classification_report # Outil pour calculer la Précision, le Rappel (Recall) et le F1-Score
from sklearn.metrics import confusion_matrix # Importation spécifique pour créer la matrice de confusion
import seaborn as sns # Bibliothèque de visualisation pour rendre la matrice de confusion plus lisible et esthétiques
import os # Bibliothèque pour interagir avec le système d'exploitation (gestion des dossiers et des fichiers d'images)


## Fonction pour afficher un certain nombre d’images d’un dossier

In [ ]:
# Import des bibliothèques nécessaires
import matplotlib.pyplot as plt
import os
from PIL import Image

# Fonction pour afficher un certain nombre d’images d’un dossier
def afficher_images(directory, class_name, n=4):
    print(f"\n--- {n} images pour {class_name} class ---")
    images_to_show = []
    # Parcours des fichiers dans le dossier
    for filename in os.listdir(directory):

        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            images_to_show.append(os.path.join(directory, filename))
        if len(images_to_show) == n:
            break

    plt.figure(figsize=(10, 5))
    for i, img_path in enumerate(images_to_show):
        img = Image.open(img_path)
        plt.subplot(1, n, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(f"{class_name} {i+1}")
        plt.axis('off')

    # Affichage final
    plt.show()


## Analyse des classes

In [ ]:
# Analyse des classes (Comptage du nombre d’images par classe)
Nbr_Nor = len(os.listdir(chemin+"/chest_xray/train/NORMAL"))
Nbr_Pneu= len(os.listdir(chemin+"/chest_xray/train/PNEUMONIA"))
print("Images NORMAL :", Nbr_Nor)
print("Images PNEUMONIA :", Nbr_Pneu)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


classes = ['NORMAL', 'PNEUMONIA']
counts = [Nbr_Nor, Nbr_Pneu]

plt.figure(figsize=(8, 6))
plt.bar(classes, counts, color=['skyblue', 'lightcoral'])
plt.xlabel('Classes')
plt.ylabel("Nombre d'images")
plt.title("Distribution des images par classe avant augmentation")
plt.show()

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

In [ ]:
# taille des images de la classe NORMAL soucre
from PIL import Image #

normal_dir = chemin + "/chest_xray/train/NORMAL"

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

# Generation methode de Rotation(ImageDataGenerator de tensorflow)

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

normal_dir = chemin + "/chest_xray/train/NORMAL"
extensions = ('.jpg', '.jpeg', '.png')

# Nombre actuel d’images
current_count = len([f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)])

target = 3875 # Objectif de 3875 images pour la classe NORMAL (équilibrage avec la classe PNEUMONIA)
generated = 0

original_images = [f for f in os.listdir(normal_dir) if f.lower().endswith(extensions)]

# Définir un répertoire accessible en écriture pour les images augmentées
save_augmented_dir = '/content/normal_rotation_augmente'
os.makedirs(save_augmented_dir, exist_ok=True)

for img_name in original_images:

    if current_count + generated >= target:
        break

    img_path = os.path.join(normal_dir, img_name)
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = x.reshape((1,) + x.shape)

    for batch in datagen.flow(
            x,
            batch_size=1,
            save_to_dir=save_augmented_dir, # Modifié en répertoire accessible en écriture
            save_prefix='aug',
            save_format='jpeg'):

        generated += 1

        if current_count + generated >= target:
            break

print("Total NORMAL :", current_count + generated)

## nombres d'image generer

In [ ]:
# Verication du nombre d'image generer
Nbr_Nor = len(os.listdir("/content/normal_rotation_augmente"))
print("Images NORMAL :", Nbr_Nor)

## taille des images de la classe NORMAL augmentee


In [ ]:
# taille des images de la classe NORMAL augmentee
from PIL import Image #

normal_dir = "/content/normal_rotation_augmente" # chemin en chemin vers les images augmentee

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## Affichage des images de la classe Normal du Data Augmentation

In [ ]:
# Affichage de 5 images de la classe NORMAL augmentee
train_dir = "/content/normal_rotation_augmente"
# affichage des images de la classe NORMAL
afficher_images(train_dir, "NORMAL ROT",5)

### Étape 1 du GAN : Chargement des images et prétraitement

In [ ]:
# Étape 1 : Chargement des images et prétraitement
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

IMG_SIZE = 224  # taille des images
LATENT_DIM = 100  # dimension du vecteur latent
BATCH_SIZE = 32  # taille du batch
EPOCHS = 1000  # nombre d’époques

normal_dir = chemin + "/chest_xray/train/NORMAL"  # dossier NORMAL
target = 3875  # objectif d’images
images = []  # stockage des images

# parcourir les fichiers du dossier
for file in os.listdir(normal_dir):

    # vérifier que le fichier est une image
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = load_img(os.path.join(normal_dir, file), target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale")  # chargement + resize + grayscale
        img = img_to_array(img)  # conversion numpy
        images.append(img)  # ajout à la liste

images = np.array(images, dtype=np.float32)  # conversion en array

# vérifier qu’il y a des images
if len(images) == 0:
    raise ValueError(f"Aucune image trouvée dans {normal_dir}")

images = (images - 127.5) / 127.5  # normalisation [-1,1]

### Étape 2 du GAN — Générateur (sortie : 224x224x1)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 2 — Générateur (sortie : 224x224x1)
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),  # vecteur latent en entrée

        layers.Dense(7 * 7 * 256, use_bias=False),  # projection dense initiale
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation

        layers.Reshape((7, 7, 256)),  # reshape en feature map

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # 14x14
        layers.BatchNormalization(),  # normalisation
        layers.LeakyReLU(),  # activation

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),  # 28x28
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, 5, strides=2, padding="same", use_bias=False),  # 56x56
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(16, 5, strides=2, padding="same", use_bias=False),  # 112x112
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", use_bias=False, activation="tanh")  # 224x224x1
    ])
    return model

### Étape 3 du GAN — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)

In [ ]:
# Étape 3 — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),  # image en entrée

        layers.Conv2D(32, 5, strides=2, padding="same"),  # réduction taille
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(128, 5, strides=2, padding="same"),  # extraction features
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Conv2D(256, 5, strides=2, padding="same"),  # features profondes
        layers.BatchNormalization(),  # stabilisation (optionnel)
        layers.LeakyReLU(),
        layers.Dropout(0.3),

        layers.Flatten(),  # mise à plat
        layers.Dense(1)  # sortie (réel/faux)
    ])
    return model

### Étape 4 du GAN — Compilation

In [ ]:
# Étape 4 — Compilation
generator = build_generator()  # création du générateur
discriminator = build_discriminator()  # création du discriminateur

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)  # fonction de perte

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)  # le générateur veut tromper le discriminateur

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output)  # perte sur vraies images avec label smoothing
    # real_loss = cross_entropy(tf.ones_like(real_output), real_output)  # version sans label smoothing
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)  # perte sur fausses images
    return real_loss + fake_loss  # perte totale du discriminateur

gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur du générateur
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur du discriminateur

### Étape 5 du GAN — Boucle d'entraînement

In [ ]:
# Étape 5 — Boucle d'entraînement
dataset = (
    tf.data.Dataset.from_tensor_slices(images)  # création dataset
    .shuffle(len(images))  # mélange
    .batch(BATCH_SIZE)  # batch (taille variable)
    .prefetch(tf.data.AUTOTUNE)  # optimisation
)

@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]  # taille batch dynamique
    noise = tf.random.normal([batch_size, LATENT_DIM])  # bruit aléatoire

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)  # génération images

        real_output = discriminator(real_images, training=True)  # prédiction réel
        fake_output = discriminator(generated_images, training=True)  # prédiction fake

        gen_loss = generator_loss(fake_output)  # perte générateur
        disc_loss = discriminator_loss(real_output, fake_output)  # perte discriminateur

    gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)  # gradients G
    gradients_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)  # gradients D

    gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))  # update G
    disc_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))  # update D

    return gen_loss, disc_loss


# boucle sur les époques
for epoch in range(1, EPOCHS + 1):
    g_losses, d_losses = [], []  # stockage pertes

    # boucle sur les batches
    for image_batch in dataset:
        g_loss, d_loss = train_step(image_batch)
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    # afficher certaines époques
    if epoch % 1000 == 0 or epoch == 800 or epoch == 500 or epoch == 300 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} - "
            f"G_loss: {tf.reduce_mean(g_losses):.4f} - "
            f"D_loss: {tf.reduce_mean(d_losses):.4f}"
        )

### Étape 6 du GAN — Générer les images manquantes

In [ ]:
# Étape 6 — Générer les images manquantes
current_count = len(images)  # nombre actuel d’images
to_generate = max(0, target - current_count)  # nombre à générer

# vérifier s’il faut générer des images
if to_generate == 0:
    print(f"Rien à générer. Images actuelles: {current_count}, cible: {target}")
else:
    noise = tf.random.normal([to_generate, LATENT_DIM])  # bruit aléatoire
    generated_images = generator(noise, training=False)  # génération images

    generated_images = ((generated_images + 1.0) * 127.5).numpy()  # retour [0,255]
    generated_images = np.clip(generated_images, 0, 255).astype(np.uint8)  # nettoyage valeurs

    gan_output_dir = '/content/normal_gan_augmented'  # dossier de sortie
    os.makedirs(gan_output_dir, exist_ok=True)  # création dossier

    # sauvegarder chaque image générée
    for i in range(to_generate):
        out_path = os.path.join(gan_output_dir, f"normal_gan_{i:05d}.png")
        save_img(out_path, generated_images[i], scale=False)  # sauvegarde image

    print("Nouvelles images générées pour la classe NORMAL :", to_generate)

### taille des images de la classe NORMAL augmentée par GAN

In [ ]:
# taille des images de la classe NORMAL augmentée par GAN
from PIL import Image  # gestion des images

normal_dir = "/content/normal_gan_augmented"  # dossier images générées
img_name = os.listdir(normal_dir)[0]  # première image du dossier
img_path = os.path.join(normal_dir, img_name)  # chemin complet
img = Image.open(img_path)  # ouverture image

print("Nom image NORMAL:", img_name)  # affichage nom
print("Taille image NORMAL (largeur, hauteur) :", img.size)  # affichage dimensions

In [ ]:
# Verication du nombre d'image generer
Nbr_Nor = len(os.listdir("/content/normal_gan_augmented"))
print("Nombre d'images de la classe NORMAL generer :", Nbr_Nor)

In [ ]:
# Affichage de 5 images de la classe NORMAL augmentée par GAN
train_dir = "/content/normal_gan_augmented"

# affichage des images de la classe NORMAL

afficher_images(train_dir, "NORMAL GAN",5)

## <center> Augmentation par la methode GAN des images de la classes PNEUMONIA pour atteindre 6417 images</center>

### Etape 1 GAN2: Chargement images PNEUMONIA

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# paramètres
IMG_SIZE = 224  # taille images
LATENT_DIM = 100  # vecteur latent
BATCH_SIZE = 32  # batch size
EPOCHS = 1000  # époques

pneumonia_dir = chemin + "/chest_xray/train/PNEUMONIA"  # dossier PNEUMONIA
target = 6417  # objectif images

images = []  # stockage images

# parcourir les fichiers
for file in os.listdir(pneumonia_dir):

    # vérifier que c’est une image
    if file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = load_img(os.path.join(pneumonia_dir, file), target_size=(IMG_SIZE, IMG_SIZE), color_mode="grayscale")  # chargement + resize
        img = img_to_array(img)  # conversion numpy
        images.append(img)  # ajout liste

images = np.array(images, dtype=np.float32)  # conversion array

# vérifier présence images
if len(images) == 0:
    raise ValueError(f"Aucune image trouvée dans {pneumonia_dir}")

images = (images - 127.5) / 127.5  # normalisation [-1,1]

### Étape 2 GAN2 — Générateur

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 2 — Générateur (sortie : 224x224x1)
def build_generator():
    model = tf.keras.Sequential([
        layers.Input(shape=(LATENT_DIM,)),  # entrée bruit

        layers.Dense(7 * 7 * 256, use_bias=False),  # projection initiale
        layers.BatchNormalization(),  # normalisation
        layers.LeakyReLU(),  # activation

        layers.Reshape((7, 7, 256)),  # reshape feature map

        layers.Conv2DTranspose(128, 5, strides=2, padding="same", use_bias=False),  # 14x14
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),  # 28x28
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(32, 5, strides=2, padding="same", use_bias=False),  # 56x56
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(16, 5, strides=2, padding="same", use_bias=False),  # 112x112
        layers.BatchNormalization(),
        layers.LeakyReLU(),

        layers.Conv2DTranspose(1, 5, strides=2, padding="same", use_bias=False, activation="tanh")  # 224x224x1
    ])
    return model

### Étape 3 GAN2 — Discriminateur

## taille des images de la classe PNEUMONIA augmentée par GAN2

In [ ]:
# imports
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import load_img, img_to_array, save_img

# Étape 3 — Discriminateur (entrée : IMG_SIZE x IMG_SIZE x 1)
def build_discriminator():
    model = tf.keras.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),  # image en entrée
        layers.Conv2D(32, 5, strides=2, padding="same"),  # extraction features
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(128, 5, strides=2, padding="same"),  # extraction profonde
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Conv2D(256, 5, strides=2, padding="same"),  # features avancées
        layers.BatchNormalization(),  # stabilisation
        layers.LeakyReLU(),  # activation
        layers.Dropout(0.3),  # régularisation

        layers.Flatten(),  # mise à plat
        layers.Dense(1)  # sortie finale
    ])
    return model

### Étape 4 GAN2 — Compilation

In [ ]:
# Étape 4 — Compilation
generator = build_generator()  # init générateur
discriminator = build_discriminator()  # init discriminateur

cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)  # loss binaire

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)  # objectif tromper D

def discriminator_loss(real_output, fake_output):
    # utiliser label smoothing (0.9 au lieu de 1)
    real_loss = cross_entropy(tf.ones_like(real_output)*0.9, real_output)
    # real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)  # faux = 0
    return real_loss + fake_loss  # perte totale

gen_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur G
disc_optimizer = tf.keras.optimizers.Adam(1e-4, beta_1=0.5)  # optimiseur D


### Étape 5 GAN2 — Boucle d'entraînement

In [ ]:
  # Étape 5 — Boucle d'entraînement
dataset = (
    tf.data.Dataset.from_tensor_slices(images)  # créer dataset
    .shuffle(len(images))  # mélanger les images
    .batch(BATCH_SIZE)  # créer les batches
    .prefetch(tf.data.AUTOTUNE)  # optimiser le chargement
)

@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]  # taille dynamique du batch
    noise = tf.random.normal([batch_size, LATENT_DIM])  # générer le bruit

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)  # générer des images
        real_output = discriminator(real_images, training=True)  # sortie vraies images
        fake_output = discriminator(generated_images, training=True)  # sortie fausses images
        gen_loss = generator_loss(fake_output)  # perte générateur
        disc_loss = discriminator_loss(real_output, fake_output)  # perte discriminateur

    gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)  # gradients générateur
    gradients_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)  # gradients discriminateur
    gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))  # mise à jour générateur
    disc_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))  # mise à jour discriminateur

    return gen_loss, disc_loss

# parcourir les époques
for epoch in range(1, EPOCHS + 1):
    g_losses, d_losses = [], []  # stocker les pertes

    # parcourir les batches
    for image_batch in dataset:
        g_loss, d_loss = train_step(image_batch)
        g_losses.append(g_loss)
        d_losses.append(d_loss)

    # afficher les pertes à certaines époques
    if epoch % 1000 == 0 or epoch == 800 or epoch == 500 or epoch == 300 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} - "
            f"G_loss: {tf.reduce_mean(g_losses):.4f} - "
            f"D_loss: {tf.reduce_mean(d_losses):.4f}"
        )

### Étape 6 GAN2 — Générer les images manquantes de PNEUMONIA

In [ ]:
# Étape 6 — Générer les images manquantes de PNEUMONIA
current_count = len(images)  # nombre actuel
to_generate = max(0, target - current_count)  # nombre à générer

# vérifier s’il faut générer
if to_generate == 0:
    print(f"Rien à générer. Images actuelles: {current_count}, cible: {target}")
else:
    noise = tf.random.normal([to_generate, LATENT_DIM])  # bruit aléatoire
    generated_images = generator(noise, training=False)  # génération images

    generated_images = ((generated_images + 1.0) * 127.5).numpy()  # [-1,1] -> [0,255]
    generated_images = np.clip(generated_images, 0, 255).astype(np.uint8)  # nettoyage valeurs

    gan_output_dir = '/content/pneumonia_gan_augmented'  # dossier sortie
    os.makedirs(gan_output_dir, exist_ok=True)  # créer dossier

    # sauvegarder les images
    for i in range(to_generate):
        out_path = os.path.join(gan_output_dir, f"pneum_gan_{i:05d}.png")
        save_img(out_path, generated_images[i], scale=False)  # sauvegarde

    print("Nouvelles images générées pour la classe PNEUMONIA :", to_generate)

## taille des images de la classe PNEUMONIA augmentée par GAN2

In [ ]:
# taille des images de la classe PNEUMONIA augmentée par GAN
from PIL import Image  # gestion images

normal_dir = "/content/pneumonia_gan_augmented"  # dossier images GAN

img_name = os.listdir(normal_dir)[0]  # première image
img_path = os.path.join(normal_dir, img_name)  # chemin complet

img = Image.open(img_path)  # ouvrir image

print("Nom image PNEUMONIA:", img_name)  # afficher nom
print("Taille image PNEUMONIA GAN (largeur, hauteur) :", img.size)  # afficher taille

In [ ]:
# nombre d'image generer pour la classe PNEUMONIA
Nbr_Nor = len(os.listdir("/content/pneumonia_gan_augmented"))
print("Nombre d'images de la classe PNEUMONIA generer :", Nbr_Nor)

In [ ]:
# Affichage de 5 images de la classe PNEUMONIA augmentée par GAN
train_dir = "/content/pneumonia_gan_augmented"

# affichage des images de la classe PNEUMONIA

afficher_images(train_dir, "PNEUMONIA GAN",5)

## Redimensionner et Sauvegarder les Images NORMAL Originales<br>Créer le répertoire `/content/normal_224`, parcourir les images NORMAL originales, les redimensionner à 224x224 et les sauvegarder dans ce nouveau répertoire.


### Redimensionnement images NORMAL (224x224)

In [ ]:
import os
from PIL import Image  # gestion images

original_normal_dir = chemin + "/chest_xray/train/NORMAL"  # dossier source
resized_normal_dir = '/content/normal_224'  # dossier destination

os.makedirs(resized_normal_dir, exist_ok=True)  # créer dossier si absent

IMG_SIZE = (224, 224)  # taille cible
processed_count = 0  # compteur

print(f"Redimensionnement des images de {original_normal_dir} vers {resized_normal_dir}")

# parcourir les fichiers
for filename in os.listdir(original_normal_dir):

    # vérifier que c’est une image
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path_source = os.path.join(original_normal_dir, filename)  # chemin source
        img_path_destination = os.path.join(resized_normal_dir, filename)  # chemin destination

        # gérer erreurs lecture image
        try:
            img = Image.open(img_path_source)  # ouvrir image
            img_resized = img.resize(IMG_SIZE)  # redimensionner
            img_resized.save(img_path_destination)  # sauvegarder
            processed_count += 1  # incrémenter compteur
        except Exception as e:
            print(f"Erreur lors du traitement de l'image {filename}: {e}")  # afficher erreur

print(f"Terminé. {processed_count} images NORMAL redimensionnées et sauvegardées dans {resized_normal_dir}.")

In [ ]:
# nombre d'images originales de la classe NORMAL redimensionnées
Nbr_Nor = len(os.listdir("/content/normal_224"))
print("Images NORMAL :", Nbr_Nor)

In [ ]:
# taille des images dans le dossier normal_224
from PIL import Image #

normal_dir = "/content/normal_224" # chemin vers le dossier normal_224

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## Redimensionner et Sauvegarder les Images PNEUMONIA Originales<br>Créer le répertoire `/content/pneumonia_224`, parcourir les images PNEUMONIA originales, les redimensionner à 224x224 et les sauvegarder dans ce nouveau répertoire.


### Redimensionnement images PNEUMONIA (224x224)

In [ ]:
import os
from PIL import Image  # gestion images

original_pneumonia_dir = chemin + "/chest_xray/train/PNEUMONIA"  # dossier source
resized_pneumonia_dir = '/content/pneumonia_224'  # dossier destination

os.makedirs(resized_pneumonia_dir, exist_ok=True)  # créer dossier

IMG_SIZE = (224, 224)  # taille cible
processed_count = 0  # compteur

print(f"Redimensionnement des images de {original_pneumonia_dir} vers {resized_pneumonia_dir}")

# parcourir les fichiers
for filename in os.listdir(original_pneumonia_dir):

    # vérifier que c’est une image
    if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_path_source = os.path.join(original_pneumonia_dir, filename)  # chemin source
        img_path_destination = os.path.join(resized_pneumonia_dir, filename)  # chemin destination

        # gérer erreurs traitement image
        try:
            img = Image.open(img_path_source)  # ouvrir image
            img_resized = img.resize(IMG_SIZE)  # redimensionner
            img_resized.save(img_path_destination)  # sauvegarder
            processed_count += 1  # incrémenter compteur
        except Exception as e:
            print(f"Erreur lors du traitement de l'image {filename}: {e}")  # afficher erreur

print(f"Terminé. {processed_count} images PNEUMONIA redimensionnées et sauvegardées dans {resized_pneumonia_dir}.")

In [ ]:
# Nombre d'images originales de la classe PNEUMONIA redimensionnées
Nbr_Pneu_resized = len(os.listdir("/content/pneumonia_224"))
print("nombre d'images PNEUMONIA redimensionnées :", Nbr_Pneu_resized)


In [ ]:
# taille des images dans le dossier pneumonia_224
from PIL import Image #

normal_dir = "/content/pneumonia_224" # chemin vers le dossier pneumonia_224

img_name = os.listdir(normal_dir)[0]  # prend la première image
img_path = os.path.join(normal_dir, img_name)

img = Image.open(img_path)

print("Nom :", img_name)
print("Taille (largeur, hauteur) :", img.size)

## creation du jeux de donnee final fusionné

## <center> Fussion des images GAN,DATA AUGMENTATION et le classe minoritaire dans un seul dossier(dossier NORMAL et dossier PNEUMONIA) Dans google drive </center>

In [ ]:
import os
import shutil

chemin_sortie = "/content/drive/MyDrive/Mes_Projet/Projet"  # dossier sortie
merged_base_dir = chemin_sortie + '/Dataset_E'  # dataset final

merged_normal_dir = os.path.join(merged_base_dir, 'NORMAL')  # dossier NORMAL
merged_pneumonia_dir = os.path.join(merged_base_dir, 'PNEUMONIA')  # dossier PNEUMONIA

os.makedirs(merged_normal_dir, exist_ok=True)  # créer NORMAL
os.makedirs(merged_pneumonia_dir, exist_ok=True)  # créer PNEUMONIA
print(f"Dossiers créés : {merged_normal_dir} et {merged_pneumonia_dir}")

# sources NORMAL
normal_rotation_aug_dir = '/content/normal_rotation_augmente'
normal_gan_aug_dir = '/content/normal_gan_augmented'
normal_original_resized_dir = '/content/normal_224'

# sources PNEUMONIA
pneumonia_gan_aug_dir = '/content/pneumonia_gan_augmented'
pneumonia_original_resized_dir = '/content/pneumonia_224'

# copier images avec gestion doublons
def copy_images(source_dir, destination_dir):
    count = 0  # compteur

    # vérifier existence dossier source
    if os.path.exists(source_dir):

        # parcourir fichiers
        for filename in os.listdir(source_dir):

            # vérifier que c’est une image
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                source_path = os.path.join(source_dir, filename)  # chemin source
                destination_path = os.path.join(destination_dir, filename)  # chemin destination

                base_name, extension = os.path.splitext(filename)  # séparer nom/ext
                counter = 1  # compteur renommage

                # gérer doublons
                while os.path.exists(destination_path):
                    new_filename = f"{base_name}_{counter}{extension}"
                    destination_path = os.path.join(destination_dir, new_filename)
                    counter += 1

                shutil.copy(source_path, destination_path)  # copier fichier
                count += 1  # incrémenter

    return count


print("\n--- Copie des images NORMAL ---")
count_normal_rot = copy_images(normal_rotation_aug_dir, merged_normal_dir)  # rotation
print(f"{count_normal_rot} images copiées depuis {normal_rotation_aug_dir}")

count_normal_gan = copy_images(normal_gan_aug_dir, merged_normal_dir)  # GAN
print(f"{count_normal_gan} images copiées depuis {normal_gan_aug_dir}")

count_normal_orig = copy_images(normal_original_resized_dir, merged_normal_dir)  # originales
print(f"{count_normal_orig} images copiées depuis {normal_original_resized_dir}")


print("\n--- Copie des images PNEUMONIA ---")
count_pneumonia_gan = copy_images(pneumonia_gan_aug_dir, merged_pneumonia_dir)  # GAN
print(f"{count_pneumonia_gan} images copiées depuis {pneumonia_gan_aug_dir}")

count_pneumonia_orig = copy_images(pneumonia_original_resized_dir, merged_pneumonia_dir)  # originales
print(f"{count_pneumonia_orig} images copiées depuis {pneumonia_original_resized_dir}")


total_normal_images = len(os.listdir(merged_normal_dir))  # total NORMAL
total_pneumonia_images = len(os.listdir(merged_pneumonia_dir))  # total PNEUMONIA

print(f"\nTotal d'images dans {merged_normal_dir}: {total_normal_images}")
print(f"Total d'images dans {merged_pneumonia_dir}: {total_pneumonia_images}")

## Création du dans google drive du dossier Dataset_D (qui stocke les images originales uniquement)

In [ ]:
import os
import shutil

chemin_sortie = "/content/drive/MyDrive/Mes_Projet/Projet"  # dossier sortie
merged_base_dir_D = chemin_sortie + '/Dataset_D'  # dataset D

merged_normal_dir_D = os.path.join(merged_base_dir_D, 'NORMAL')  # dossier NORMAL
merged_pneumonia_dir_D = os.path.join(merged_base_dir_D, 'PNEUMONIA')  # dossier PNEUMONIA

os.makedirs(merged_normal_dir_D, exist_ok=True)  # créer NORMAL
os.makedirs(merged_pneumonia_dir_D, exist_ok=True)  # créer PNEUMONIA
print(f"Dossiers créés : {merged_normal_dir_D} et {merged_pneumonia_dir_D}")

normal_original_resized_dir = '/content/normal_224'  # source NORMAL
pneumonia_original_resized_dir = '/content/pneumonia_224'  # source PNEUMONIA

# copier images avec gestion doublons
def copy_images(source_dir, destination_dir):
    count = 0  # compteur

    # vérifier existence dossier source
    if os.path.exists(source_dir):

        # parcourir fichiers
        for filename in os.listdir(source_dir):

            # vérifier que c’est une image
            if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
                source_path = os.path.join(source_dir, filename)  # chemin source
                destination_path = os.path.join(destination_dir, filename)  # chemin destination

                base_name, extension = os.path.splitext(filename)  # séparer nom/ext
                counter = 1  # compteur renommage

                # gérer doublons
                while os.path.exists(destination_path):
                    new_filename = f"{base_name}_{counter}{extension}"
                    destination_path = os.path.join(destination_dir, new_filename)
                    counter += 1

                shutil.copy(source_path, destination_path)  # copier
                count += 1  # incrémenter

    return count


print("\n--- Copie des images NORMAL ---")
count_normal_D = copy_images(normal_original_resized_dir, merged_normal_dir_D)  # copie NORMAL
print(f"{count_normal_D} images copiées depuis {normal_original_resized_dir} vers {merged_normal_dir_D}")


print("\n--- Copie des images PNEUMONIA ---")
count_pneumonia_D = copy_images(pneumonia_original_resized_dir, merged_pneumonia_dir_D)  # copie PNEUMONIA
print(f"{count_pneumonia_D} images copiées depuis {pneumonia_original_resized_dir} vers {merged_pneumonia_dir_D}")


total_normal_images_D = len(os.listdir(merged_normal_dir_D))  # total NORMAL
total_pneumonia_images_D = len(os.listdir(merged_pneumonia_dir_D))  # total PNEUMONIA

print(f"\nTotal d'images dans {merged_normal_dir_D}: {total_normal_images_D}")
print(f"Total d'images dans {merged_pneumonia_dir_D}: {total_pneumonia_images_D}")

## Histogramme des couleurs avant contraste

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os

# chemin vers ton dataset
dataset_path = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E/PNEUMONIA"

# liste des images
images = os.listdir(dataset_path)[10:14]

plt.figure(figsize=(12,8))

for i, img_name in enumerate(images):

    img_path = os.path.join(dataset_path, img_name)

    image = cv2.imread(img_path, 0)  # lecture en niveaux de gris

    plt.subplot(2,2,i+1)
    plt.hist(image.ravel(), 256, [0,256])
    plt.title(f"Histogramme {img_name}")

plt.tight_layout()
plt.show()

# contraster les images d'entrainement du dataset equilibrer (Dataset_E)

In [ ]:
import os
import cv2  # OpenCV

input_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E"  # dossier source
output_base_folder = "/content/drive/MyDrive/Mes_Projet/Projet/Dataset_E_CONT"  # dossier sortie

os.makedirs(output_base_folder, exist_ok=True)  # créer dossier sortie

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))  # initialiser CLAHE

print(f"Applying CLAHE to images from {input_base_folder} and saving to {output_base_folder}")

# parcourir les classes
for class_name in os.listdir(input_base_folder):
    input_class_path = os.path.join(input_base_folder, class_name)  # chemin entrée
    output_class_path = os.path.join(output_base_folder, class_name)  # chemin sortie

    # vérifier que c’est un dossier
    if os.path.isdir(input_class_path):
        os.makedirs(output_class_path, exist_ok=True)  # créer dossier classe
        print(f"Processing images in {input_class_path}...")

        # parcourir les images
        for img_name in os.listdir(input_class_path):

            # vérifier que c’est une image
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path_source = os.path.join(input_class_path, img_name)  # source
                img_path_destination = os.path.join(output_class_path, img_name)  # destination

                img = cv2.imread(img_path_source, cv2.IMREAD_GRAYSCALE)  # lire image

                # vérifier chargement image
                if img is not None:
                    enhanced = clahe.apply(img)  # appliquer CLAHE
                    cv2.imwrite(img_path_destination, enhanced)  # sauvegarder
                else:
                    print(f"Warning: Could not read image {img_path_source}. Skipping.")
    else:
        print(f"Warning: Skipping non-directory item in {input_base_folder}: {class_name}")

print("CLAHE application complete.")